# 10주차 실습 — 안전한 데이터 수정과 트랜잭션

- **데이터**: `studies` 에서 100행을 떼어 만든 연습용 표 `연습_시험`. 원본은 그대로 둡니다.
- **오늘 배우는 것**: ① INSERT · UPDATE · DELETE의 기본 모양을 읽고 쓸 수 있다 ② WHERE를 빠뜨렸을 때 무슨 일이 생기는지 설명할 수 있다 ③ 트랜잭션이 왜 필요한지 병원 예로 설명할 수 있다 ④ 안전한 수정 절차 여섯 단계를 말할 수 있다
- **점수는 없습니다.**

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` · `-- S1` 같은 문항 이름 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
_pd.set_option("display.max_columns", None)     # SELECT * 로 열 이름을 볼 때 모든 열을 보여 줌
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

ANSWERS = _load_answers("eNq9lUtvElEUgP/KCRvahDQdHgImLmpLbBdFaTFdGNNMYVASBAOoi6bJVKYGC0YwTJlWaGmC9iELpKg0wT8098x/8FyEOp3WJkYtC7ice+7r+86dWbWFBNtNWLVl4o/o1+aRYjcmfWJU9IpOmwNsqd/E06kXGYoL1IykErz5wJaMZJfjUd67ko5LseVsPJuQBpM8l9JiIrGcyYrZZxkekZLpVCLxREpmbQ/p7+M4NWgVvdPWT/pgbKtY74FRrmG1BMHpsN8/yT8CGGoNjK3XWO8CO+litYUqpatHrJBnheYE6O13WJdB7yqG1gNU84DVLmC+hkqNFWW2XwNsrOsnDQfoPRl3S2yzArhbpkGs0katS2vyJL1dwkKP8mjsp2Ga/q03mqKyPlqQvrBAe1IbwNTiYLaOhvUi5taNHJ9qbBxwQ2ZfZUNpAxaatHtD1VDp4U5lYgAnHZXSUpSOHxMTGYlHnmbjqaSYGIXWKBZymi1NRrx+r1dwR9xRv9mSNX6ZJdquUdUwr11Cnu/fBUQXt7oWskuzgYUAcB92jvQWR20HdnyIW2Ws9YdkcfPLaABlcRXn3TtGhuyL9xfvBYIzgRk7jHGsrR5hdlhdjP8JIpcZkc/j8gmiX3B7nStmRNb4EJHXjOhivV4F7Xy5UqHB2dnotDLgyxar9/FA5uUUDizMzwWnwsM+wQeDWi5pF4kTW1YumUewo+7QA/vQB3baYLVD9rENeieHuxujcYuBMPc0qMVcngN3/LLHoyEn4DZdkvY2N4H7bf0zqVIs/q6tst1mbRGP4JJ8TlfM75bM2qzx/6FtaS48O7MwtRQEbCrsoDgy1+hZzNAQLo+dKljN8+K/zEujOTJI6nGna7F0bXg9ZryiOyJ5Y4LgjUXOPd6t8SFe3z/BO7gEPw/X0dhen+qwxfZacHZUYG+LqCpUlFbQNEdDHZu+Oz8/Fx7nN4XSeGe9D7cDd+aC/PYY7xWsa/Q+kPXO90EdV1+xNxp1DZ5KBI3TZccbrHF6pYCQh8TLZPXvwa/9AAjucTY=")   # 자동 확인용 (정답 SQL 은 없음)

In [ ]:
#@title [2번] ▶ 수정 연습 준비 — 실행만 하세요 (Shift+Enter) { display-mode: "form" }
class _한연결:
    """%%sql 셀이 BEGIN 을 다음 셀의 COMMIT · ROLLBACK 까지 이어 가게 합니다."""
    def __init__(self, con): self.con = con
    def cursor(self): return self
    def commit(self): pass
    def close(self): pass
    def __getattr__(self, name): return getattr(self.con, name)

if "_rw" not in globals():
    _rw = _한연결(conn)
%sql _rw --alias rw

def 처음상태로():
    """연습_시험 을 처음 100행으로 되돌립니다."""
    try:
        _rw.execute("ROLLBACK")
    except Exception:
        pass
    _rw.execute("""CREATE OR REPLACE TEMP TABLE 연습_시험 AS
SELECT nct_id, brief_title, overall_status, enrollment
FROM studies
ORDER BY nct_id
LIMIT 100;""")
    print("연습_시험 을 처음 상태(100행)로 되돌렸습니다.")

print("준비 완료")

## 1. 데이터를 바꾼다는 것 — 조회와 무엇이 다른가

### 조회와 수정은 다릅니다

지금까지 배운 SELECT는 데이터를 읽기만 했습니다.

| 조회 (SELECT) | 수정 (INSERT · UPDATE · DELETE) |
|---|---|
| 데이터를 읽기만 함 | 데이터가 실제로 바뀜 |
| 잘못 써도 데이터는 그대로 | 조건을 잘못 쓰면 전체가 바뀜 |
| 다시 실행하면 됨 | 확정 후에는 되돌리기 어려움 |
| 연구자 · 분석자에게 주는 기본 권한 | 권한이 따로 필요 (3주차 GRANT) |

병원에서 분석자는 보통 읽기 권한만 받습니다. 원천 데이터는 수정하지 않습니다.

그래도 문법을 알아야 하는 이유: 데이터가 어떻게 바뀌는지 알아야 결과를 해석할 수 있기 때문입니다.

## 2. INSERT · UPDATE · DELETE — 세 가지 문장

### 연습용 사본 만들기

실습 DB는 읽기 전용이라, 고치는 연습은 사본에서 합니다.

**질문** 원본을 건드리지 않고 INSERT · UPDATE · DELETE를 연습하려면?

In [ ]:
%%sql
-- [3번] ① studies에서 100행만 떼어 연습용 표를 만듭니다 (OR REPLACE · ORDER BY 추가)
CREATE OR REPLACE TEMP TABLE 연습_시험 AS
SELECT nct_id, brief_title, overall_status, enrollment
FROM studies
ORDER BY nct_id
LIMIT 100;

In [ ]:
%%sql
-- [4번] ② 이제 이 사본만 고칩니다 → WITHDRAWN 이 COMPLETED 로
UPDATE 연습_시험
SET overall_status = 'COMPLETED'
WHERE nct_id = 'NCT00455039';

SELECT * FROM 연습_시험 WHERE nct_id = 'NCT00455039';

| 표 | 고칠 수 있나 |
|---|---|
| `studies` (원본) | 아니오 · 읽기 전용 |
| `연습_시험` (사본) | 예 · 노트북을 닫으면 사라짐 |

TEMP 표는 9강 4단계 분석표와 같은 방식입니다. 지워도 원본에는 영향이 없습니다.

**원칙** 원본은 고치지 않고 사본에서 연습합니다. 실무에서도 같습니다 (11강 CDW).

‘환자’ 표 예시는 개념을 보여 주는 가상 표입니다. 실습은 이 연습_시험 표로 합니다.

표가 꼬이면 `처음상태로()` 를 실행하세요. 연습_시험 이 처음 100행으로 돌아갑니다.

### INSERT · UPDATE · DELETE — 추가 · 수정 · 삭제의 기본 모양

```sql
-- INSERT: 새 행을 추가
INSERT INTO 환자
  (환자ID, 환자명, 생년월일)
VALUES ('P100', '김OO', '1958-03-11');

-- UPDATE: 기존 값을 바꿈
UPDATE 환자
SET 연락처 = '010-9999-8888'
WHERE 환자ID = 'P100';

-- DELETE: 행을 지움
DELETE FROM 환자
WHERE 환자ID = 'P100';
```

**UPDATE와 DELETE에는 WHERE가 거의 반드시 필요합니다.** 빠뜨리면 모든 행이 대상이 됩니다.

같은 세 문장을 연습_시험 에 해 봅니다.

In [ ]:
%%sql
-- [5번] INSERT: 새 행을 추가
INSERT INTO 연습_시험
  (nct_id, brief_title, overall_status, enrollment)
VALUES ('NCT99999999', '연습용 시험', 'RECRUITING', 50);

SELECT * FROM 연습_시험 WHERE nct_id = 'NCT99999999';

In [ ]:
%%sql
-- [6번] UPDATE: 기존 값을 바꿈 → enrollment 50 이 60 으로
UPDATE 연습_시험
SET enrollment = 60
WHERE nct_id = 'NCT99999999';

SELECT * FROM 연습_시험 WHERE nct_id = 'NCT99999999';

In [ ]:
%%sql
-- [7번] DELETE: 행을 지움 → 다시 100행
DELETE FROM 연습_시험
WHERE nct_id = 'NCT99999999';

SELECT COUNT(*) FROM 연습_시험;

### WHERE를 빠뜨리면 — 오류가 나지 않아서 더 위험합니다

실무에서 가장 유명한 사고입니다. 오류가 나지 않고 **조용히 전부 바뀝니다**.

```sql
UPDATE 환자
SET 진료과 = '내과';
-- WHERE 가 없습니다
```

→ 모든 환자의 진료과가 내과로 바뀝니다. 5만 명이면 5만 행이 한 번에 바뀝니다.

In [ ]:
%%sql
-- [8번] WHERE 를 빠뜨림 → 오류 없이 100행 모두 COMPLETED
UPDATE 연습_시험
SET overall_status = 'COMPLETED';
-- WHERE 가 없습니다

SELECT overall_status, COUNT(*) AS 시험수
FROM 연습_시험
GROUP BY overall_status
ORDER BY overall_status;

In [ ]:
처음상태로()   # [9번]

### 이렇게 확인하고 씁니다

```sql
SELECT COUNT(*) FROM 환자
WHERE 진료과 = '외과';

UPDATE 환자
SET 진료과 = '내과'
WHERE 진료과 = '외과';
```

→ 바꿀 조건을 WHERE에 적고, **같은 조건으로 먼저 세어 본 뒤** 수정합니다.

In [ ]:
%%sql
-- [10번] ① 같은 WHERE 로 먼저 세기 → 18
SELECT COUNT(*) FROM 연습_시험
WHERE overall_status = 'UNKNOWN';

In [ ]:
%%sql
-- [11번] ② 수정 → Count(바뀐 행 수)가 ①의 18 과 같은지 확인
UPDATE 연습_시험
SET overall_status = 'COMPLETED'
WHERE overall_status = 'UNKNOWN';

**병원 사례**: 처방 코드 일괄 정정 작업에서 WHERE를 빠뜨려 수개월치 처방이 한 값으로 덮여 쓰인 사고가 있었습니다.

그래서 운영 데이터베이스 수정은 혼자 하지 않고, 백업을 확인한 뒤, 승인 절차를 거쳐 진행합니다.

### 수정할 때 확인할 것 — 문장별로 특히 주의할 점

| 문장 | 확인할 것 | 안 지키면 |
|---|---|---|
| INSERT | 기본키 중복, 필수 열 누락, 타입 맞음 | 개체 무결성 위반으로 거부 |
| UPDATE | WHERE 조건, 바뀔 행 수, 바꿀 값의 타입 | 의도하지 않은 행까지 변경 |
| DELETE | WHERE 조건, 이 행을 참조하는 다른 표 | 참조 무결성 위반 또는 연쇄 삭제 |

5주차의 무결성 제약조건이 여기서 실제로 작동합니다.

### 여기까지 정리

- ① INSERT · UPDATE · DELETE 로 데이터를 바꾼다
- ② UPDATE · DELETE 에는 WHERE가 거의 반드시 필요하다
- ③ 수정 전에 같은 WHERE로 SELECT 해서 행 수를 확인한다

**잠깐 생각해 봅시다** UPDATE 문에서 WHERE를 빠뜨리면 어떻게 될까요? → 오류 없이 모든 행이 바뀝니다. 그래서 가장 위험한 실수입니다.

## 3. 트랜잭션 — 한 덩어리로 묶기

### 트랜잭션 (Transaction) — 입원 수속을 예로

**트랜잭션** = 여러 개의 수정을 하나의 덩어리로 묶은 것. 전부 되거나, 전부 안 되거나.

**입원 수속 (한 덩어리)**: ① 입원 기록을 만든다 → ② 병상을 ‘사용 중’으로 바꾼다 → ③ 수납 내역을 만든다

**②에서 오류가 나면?** 입원 기록은 있는데 병상은 비어 있는 상태. 데이터가 서로 맞지 않게 됩니다.

**그래서 묶습니다.** 하나라도 실패하면 전부 없던 일로 되돌립니다.

**ACID** — 트랜잭션이 지켜 주는 네 가지 (용어만 기억하면 됩니다): 원자성(전부 또는 전무) · 일관성(규칙 유지) · 고립성(서로 방해 안 함) · 지속성(확정되면 남음)

### 트랜잭션이 없으면 — 병원에서 실제로 곤란해지는 경우

수혈 기록과 혈액 재고를 각각 따로 수정한다고 해 봅시다.

| 중간에 실패하면 | 트랜잭션으로 묶으면 |
|---|---|
| 수혈 기록은 남았는데 재고는 그대로 | 둘 다 되거나, 둘 다 안 되거나 |
| 재고 수량이 실제와 어긋남 | 중간 상태가 남지 않음 |
| 어느 쪽이 맞는지 알 수 없음 | 실패하면 자동으로 되돌아감 |
| 나중에 사람이 찾아서 고쳐야 함 | 데이터가 항상 맞물려 있음 |

계좌 이체, 재고 관리, 예약 시스템이 모두 같은 이유로 트랜잭션을 씁니다.

**여러 표를 함께 바꿀 때는 반드시 트랜잭션으로 묶습니다.**

## 4. COMMIT과 ROLLBACK — 확정과 되돌리기

수정한 뒤 확인하고 확정하거나, 되돌립니다.

```sql
BEGIN;
UPDATE 환자
SET 연락처 = '010-9999-8888'
WHERE 환자ID = 'P100';
SELECT * FROM 환자 WHERE 환자ID = 'P100';
```

| 명령 | 언제 | 하는 일 |
|---|---|---|
| `COMMIT;` | 결과가 맞으면 | 바뀐 내용을 확정합니다. 이제 되돌릴 수 없습니다. |
| `ROLLBACK;` | 이상하면 | BEGIN 이후의 모든 변경을 없던 일로 되돌립니다. |

**중요**: COMMIT 하기 전에는 다른 사람에게 이 변경이 보이지 않습니다 (고립성).

일부 도구는 자동으로 COMMIT 됩니다(auto-commit). 수정 작업 전에 어떤 설정인지 반드시 확인하세요.

### 연습_시험 으로 해 보기

BEGIN 이 든 셀은 한 번만 실행하세요. 꼬이면 `처음상태로()` 를 실행합니다.

In [ ]:
%%sql
-- [12번] BEGIN 후 수정하고, 결과를 조회해 확인 (아직 확정 전)
BEGIN;

UPDATE 연습_시험
SET overall_status = 'COMPLETED'
WHERE nct_id = 'NCT00455039';

SELECT * FROM 연습_시험 WHERE nct_id = 'NCT00455039';

In [ ]:
%%sql
-- [13번] 결과가 맞으면 COMMIT → 확정
COMMIT;

SELECT * FROM 연습_시험 WHERE nct_id = 'NCT00455039';

In [ ]:
%%sql
-- [14번] BEGIN 뒤에 WHERE 없는 DELETE → 0행 (아직 확정 전)
BEGIN;

DELETE FROM 연습_시험;
-- WHERE 를 빠뜨렸습니다

SELECT COUNT(*) FROM 연습_시험;

In [ ]:
%%sql
-- [15번] 이상하면 ROLLBACK → 다시 100행
ROLLBACK;

SELECT COUNT(*) FROM 연습_시험;

## 5. 안전한 수정 절차와 뷰 — 실무에서 지키는 순서

### 안전한 수정 절차 — 실무에서 지키는 여섯 단계

| 단계 | 할 일 |
|---|---|
| ① 환경 확인 | 운영 데이터베이스인가, 연습용 복사본인가 |
| ② 백업 확인 | 언제 백업되었는지, 복구에 얼마나 걸리는지 |
| ③ SELECT 먼저 | 같은 WHERE로 조회해 바뀔 행 수를 확인 |
| ④ BEGIN 후 수정 | 트랜잭션 안에서 UPDATE · DELETE 실행 |
| ⑤ 결과 확인 | 바뀐 행 수와 실제 값을 다시 조회 |
| ⑥ COMMIT / ROLLBACK | 맞으면 확정, 이상하면 되돌리기 |

이 순서를 지키면 사고의 대부분을 막을 수 있습니다.

### 분석자가 실제로 지키는 규칙 — 권한과 절차가 우리를 지켜 줍니다

| 규칙 | 뜻 |
|---|---|
| ① 원천 데이터는 건드리지 않는다 | EMR · CDW의 원본 테이블은 읽기 전용으로 사용 |
| ② 분석용 결과는 따로 만든다 | 새 표를 만들거나 파일로 내보내서 작업 |
| ③ 수정이 필요하면 담당 부서에 요청한다 | 의료정보팀 · DBA를 통해 절차대로 진행 |
| ④ 연습은 복사본에서 한다 | 수업에서도 원본이 아닌 연습용 복사본으로만 시연합니다 |

“읽기 권한만 있는 것”은 불편이 아니라 보호 장치입니다.

### 뷰(View)로 자주 쓰는 조회를 저장하기 — 긴 조건을 이름 하나로 줄여 둡니다

요청: “이 조건을 매번 다시 쓰지 않게 해 주세요”

In [ ]:
%%sql
-- [16번] 뷰 만들기 → 1,353 (읽기 전용이라 TEMP VIEW)
CREATE OR REPLACE TEMP VIEW 제2형당뇨_시험 AS
SELECT DISTINCT studies.nct_id, brief_title,
                overall_status, start_date
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2';

SELECT COUNT(*) FROM 제2형당뇨_시험;

뷰는 표를 새로 만드는 것이 아니라 **조회 방법에 이름을 붙이는 것**입니다. 원본은 그대로 남습니다 (4주차 외부 스키마).

9강의 CREATE TEMP TABLE은 결과를 저장하고, VIEW는 조회 방법만 저장합니다.

### 뷰를 실무에서 쓰는 이유 — 같은 조건을 여러 사람이 쓸 때

| 이유 | 무엇이 좋아지나 |
|---|---|
| 정의를 한 곳에 | 코호트 조건이 바뀌면 뷰만 고치면 됩니다 |
| 같은 숫자를 공유 | 여러 사람이 같은 정의로 세게 됩니다 |
| 필요한 열만 보여 주기 | 이름 · 연락처를 뺀 뷰로 권한을 제한할 수 있습니다 |
| 긴 SQL을 줄이기 | 매번 JOIN과 조건을 다시 적지 않아도 됩니다 |

분석자에게는 대개 뷰를 만들 권한이 없습니다. 필요하면 데이터 담당 부서에 요청합니다.

## 6. 정리 — 용어와 요약

**10주차 용어 정리** — 이번 시간에 새로 나온 말

| 용어 | 영어 | 뜻 (한 줄) |
|---|---|---|
| 뷰 | View | 조회 방법에 이름을 붙여 둔 것. 원본 표는 그대로 |
| INSERT | — | 새 행을 추가하는 문장 |
| UPDATE | — | 기존 행의 값을 바꾸는 문장 |
| DELETE | — | 행을 지우는 문장 |
| 트랜잭션 | Transaction | 여러 수정을 하나로 묶은 처리 단위 |
| ACID | — | 원자성 · 일관성 · 고립성 · 지속성 |
| COMMIT | — | 변경을 확정하는 명령 |
| ROLLBACK | — | 변경을 없던 일로 되돌리는 명령 |
| 자동 커밋 | Auto-commit | 실행하자마자 확정되는 설정. 수정 전 확인 필요 |

## 직접 해보기 — 5문항

각 문항은 `처음상태로()` 셀부터 실행하세요. 빈칸 `___` 을 채워 실행하면 결과 아래에 ✅ / ❌ 가 나옵니다.

**열 이름 찾기** — 표의 열 이름은 `DESCRIBE` 로 봅니다 (9주차 0단계와 같은 방법). 표 이름만 바꾸면 다른 표도 볼 수 있습니다.

In [ ]:
%%sql
-- [17번] 연습_시험 의 열 이름 · 타입 보기
SELECT *
FROM (DESCRIBE 연습_시험);

**Q1.** 연습_시험 에 새 시험 한 행을 추가하세요. nct_id `'NCT99000001'`, brief_title `'당뇨 교육 연구'`, overall_status `'RECRUITING'`, enrollment `40`.

In [ ]:
처음상태로()   # [18번]

In [ ]:
%%sql
-- [19번] Q1
INSERT INTO 연습_시험 (nct_id, brief_title, overall_status, enrollment)
VALUES (___, ___, ___, ___);

-- 결과 확인 (그대로 두세요)
SELECT * FROM 연습_시험 WHERE nct_id = 'NCT99000001';

**Q2.** Q3 에서 상태가 `'SUSPENDED'` 인 시험을 바꿉니다. 바꾸기 전에 몇 행인지 먼저 세세요.

쓰는 열: `overall_status`

In [ ]:
처음상태로()   # [20번]

In [ ]:
%%sql
-- [21번] 연습_시험 의 열 이름 · 타입 보기
SELECT *
FROM (DESCRIBE 연습_시험);

In [ ]:
%%sql
-- [22번] Q2
SELECT COUNT(*) AS 시험수
FROM 연습_시험
WHERE ___;

**Q3.** Q2 와 **같은 WHERE** 로, SUSPENDED 시험의 상태를 `'TERMINATED'` 로 바꾸세요.

In [ ]:
처음상태로()   # [23번]

In [ ]:
%%sql
-- [24번] Q3
UPDATE 연습_시험
SET overall_status = ___
WHERE ___;

-- 결과 확인 (그대로 두세요)
SELECT overall_status, COUNT(*) AS 시험수
FROM 연습_시험
GROUP BY overall_status
ORDER BY overall_status;

**Q4.** 철회된 시험(`'WITHDRAWN'`)을 분석에서 빼려고 합니다. WITHDRAWN 시험만 지우세요.

쓰는 열: `overall_status`

In [ ]:
처음상태로()   # [25번]

In [ ]:
%%sql
-- [26번] 연습_시험 의 열 이름 · 타입 보기
SELECT *
FROM (DESCRIBE 연습_시험);

In [ ]:
%%sql
-- [27번] Q4
DELETE FROM 연습_시험
WHERE ___;

-- 결과 확인 (그대로 두세요)
SELECT overall_status, COUNT(*) AS 시험수
FROM 연습_시험
GROUP BY overall_status
ORDER BY overall_status;

**Q5.** 아래 준비 셀은 BEGIN 후 **WHERE 없는 UPDATE** 를 실행합니다 (모두 COMPLETED, 아직 확정 전). `___` 에 명령을 넣어 되돌리세요.

In [ ]:
# [28번]
# Q5 준비: BEGIN 후 WHERE 없는 UPDATE
처음상태로()
%sql BEGIN;
%sql UPDATE 연습_시험 SET overall_status = 'COMPLETED';
%sql SELECT overall_status, COUNT(*) AS 시험수 FROM 연습_시험 GROUP BY overall_status;

In [ ]:
%%sql
-- [29번] Q5
___;

-- 결과 확인 (그대로 두세요)
SELECT overall_status, COUNT(*) AS 시험수
FROM 연습_시험
GROUP BY overall_status
ORDER BY overall_status;

## 오늘 배운 것

- **UPDATE · DELETE 에는 WHERE.** 빠뜨리면 모든 행이 바뀐다
- **같은 WHERE로 SELECT 먼저** 해서 바뀔 행 수를 확인한다
- 여러 수정은 **트랜잭션**으로 묶고, 확인 후 **COMMIT 또는 ROLLBACK**

다음 시간: 11강. CDW 구조와 분석 업무 흐름 — 병원에서 데이터를 받는 과정

In [ ]:
진행확인()   # [30번]